In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from pathlib import Path
import pickle
import operator

import IPython.display as ipydisplay
import numpy as np
import scipy as sp
from openmm.app import PDBFile
from openbabel import pybel
import pandas as pd
from pdbfixer import PDBFixer

import sciapi
import scids
import scifile
import scishow
import caddpy

Define paths for storing intermediate results:

In [ ]:
project_id = "1AQ1"

cache_dir = Path(f".tmp/{project_id}")
assert " " not in str(cache_dir), "autogrid does not accept whitespace in paths"
pdb_filepath_raw = cache_dir / "receptor_raw.pdb"
pdb_filepath_fixed = cache_dir / "receptor_fixed.pdb"
pdb_filepath_apo = cache_dir / "receptor_fixed_apo.pdb"
pdbqt_filepath = cache_dir / "receptor.pdbqt"
pockets_filepath = cache_dir / "pockets.pkl"
autogrid_dirpath = cache_dir/ "autogrid"
autogrid_common_path = autogrid_dirpath / project_id
gpf_filepath = autogrid_common_path.with_suffix(".gpf")
cache_dir.mkdir(exist_ok=True, parents=True)
autogrid_dirpath.mkdir(exist_ok=True, parents=True)

## Structure Preparation

Obtain a PDB file:

In [ ]:
pdb_id = project_id

if not pdb_filepath_raw.is_file():
    pdb_file_content = sciapi.pdb.file.entry(pdb_id=pdb_id, file_format="pdb")
    pdb_filepath_raw.write_bytes(pdb_file_content)

Fix the PDB file:

In [ ]:
if not pdb_filepath_fixed.is_file():
    fixer = PDBFixer(filename=str(pdb_filepath_raw))
    fixer.findMissingResidues()
    print("Missing residues:", fixer.missingResidues)
    fixer.findNonstandardResidues()
    print("Nonstandard residues:", fixer.nonstandardResidues)
    fixer.replaceNonstandardResidues()
    fixer.findMissingAtoms()
    print("Missing atoms:", fixer.missingAtoms)
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_filepath_fixed, 'w'))

We store one fixed PDB with ligands, and one without.
The one with ligands is used later as reference for visualization and validation.
The one without ligands is used to create a PDBQT file for AutoGrid:

In [ ]:
if not pdb_filepath_apo.is_file():
    fixer = PDBFixer(filename=str(pdb_filepath_fixed))
    fixer.removeHeterogens(False)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_filepath_apo, 'w'))

Create the PDBQT file from the cleaned and fixed PDB file:

In [ ]:
if not pdbqt_filepath.is_file():
    molecule = next(pybel.readfile("pdb", str(pdb_filepath_apo)))
    molecule.calccharges("gasteiger")
    molecule.write(
        format="pdbqt",
        filename=str(pdbqt_filepath),
        overwrite=True,
        opt={"r": None, "n": None, "p": None},
    )


Read the created PDBQT file (used later for atom type references):

In [ ]:
pdbqt_file = scifile.autodock_pdbqt.read(pdbqt_filepath)

## Binding Site Detection

In [ ]:
pdb = scifile.pdb.read(pdb_filepath_fixed)

In [ ]:
center = pdb.atom[pdb.atom["res_name"] == "STU"][["x", "y", "z"]].mean().to_numpy()
center

In [ ]:
grid = scids.grid.from_size_spacing_anchor(
    size=(16, 16, 16),
    spacings=0.6,
    anchor="center",
    anchor_coord=center,
)
grid

## Energy Calculation

In [ ]:
ligand_types = ("C", "HD", "OA")
field_names = ligand_types + ("e", "d")
mif = caddpy.mif.autogrid.from_pdbqt(
    files=pdbqt_filepath,
    grid=grid,
    ligand_types=ligand_types,
    output_dir=autogrid_dirpath,
)

In [ ]:
sys = caddpy.chemsys.from_pdb(pdb_filepath_apo)
field = sys.toxelate(grid=grid)

In [ ]:
ligsite = caddpy.pocket.ligsite.LigSite(field, directions=(1, 3))

In [ ]:
buriedness = ligsite.psp_count >= 4

In [ ]:
vacancy = mif.tensor[1] <= 0.6

In [ ]:
site = np.logical_and(buriedness, vacancy)

In [ ]:
interactions = caddpy.interaction.from_pdb(pdb_filepath_fixed)

In [ ]:
nw = scishow.nglview.NGLWidget()
nw.add_component(str(pdb_filepath_fixed))
# Add the pocket
nw.add_spheres(
        coords=grid.coordinates[site],
        radii=grid.spacings[0]/2,
        name="pocket",
        representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=False)
    )
nw.add_box(grid.lower_bounds, grid.upper_bounds)

In [ ]:
dist, idx_self, idx_point = sys.trajectory.distance_matrix_sparse(
    points=grid.coordinates[site],
    max_distance=3,
)

In [ ]:
idx_point

In [ ]:
field_colors = {
    "c":  [255, 165, 0],     # Hydrophobic (Orange)
    "a":  [128, 0, 128],     # Aromatic (Purple)
    "hd": [0, 255, 255],     # H-bond Donor (Cyan)
    "n":  [0, 0, 255],       # Non-Hbonding Nitrogen (Blue)
    "na": [30, 144, 255],    # H-bonding Nitrogen (Dodger Blue)
    "oa": [0, 128, 0],       # H-bonding Oxygen (Green)
    "sa": [255, 20, 147],    # H-bonding Sulphur (Deep Pink)
    "e":  [255, 0, 0],       # Electrostatic (Red)
    "d":  [255, 215, 0],     # Desolvation (Gold)
    "pi": [255, 0, 0],
    "ni": [0, 0, 255],
}

In [ ]:
np.count_nonzero(np.logical_and(site, mif.tensor[-2] < -1))

## Feature Selection

Define selection criteria:

In [ ]:
field_names = ligand_types + ("PI", "NI")
field_indices = (0, 1, 2, 3, 3)
field_cutoffs = (-0.4, -0.35, -0.6, -1, 1)
cutoff_operators = (operator.lt, operator.lt, operator.lt, operator.lt, operator.gt)

Generate features:

In [ ]:
masks = []
for field_name, field_idx, field_cutoff, cutoff_operator in zip(field_names, field_indices, field_cutoffs, cutoff_operators):
    mask_field = cutoff_operator(mif.tensor[field_idx], field_cutoff)
    mask_final = np.logical_and(mask_field, site)
    masks.append(mask_final)

In [ ]:
for field_name, mask in zip(field_names, masks):
    nw.add_spheres(
        coords=grid.coordinates[mask],
        colors=field_colors[field_name.lower()],
        radii=grid.spacings[0]/2,
        name=field_name,
        representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
    )

In [ ]:
nw.display(gui=True)

In [ ]:
interactions.display(nw, vis=("ligand", "water"))

## Visualization

All fields and features are added (hidden by default) under their respective atom type names. You can find them in the GUI. Toggle visibility for each field and vary the iso-level to analyze the field visually.

In [ ]:
nw = scishow.nglview.NGLWidget()
nw.add_component(str(pdb_filepath_fixed))
# Add the pocket
nw.add_volume(
    pocket.data,
    basis=pocket.grid_vectors,
    origin=pocket.grid_origin,
    name="pocket",
    representation_params=scishow.nglview.SurfaceRepresentationParameters(lazy=True, opacity=1, contour=True)
)
# Add the fields
for field_idx, field_name in enumerate(ligand_types + ("e", "d")):
    field_ = tensor[field_idx]
    nw.add_volume(
        field_,
        basis=mif.grid.unit_vectors,
        origin=mif.grid.lower_bounds,
        name=field_name,
        representation_params=scishow.nglview.SurfaceRepresentationParameters(
            lazy=True, opacity=1, visible=False, contour=True, isolevel=0, isolevel_type="value",
            isolevel_scroll=True, use_worker=True, color_value=f"rgb({", ".join(map(str,field_colors[field_name.lower()]))})"
        )
    )


voxel_volume = np.prod(grid.spacings)

for feat_idx, feature in enumerate(features):
    nw.add_spheres(
        coords=feature["coords"],
        colors=field_colors[feature["type"].lower()],
        radii=grid.spacings[0]/2,
        name=f"{feature["type"]}_{feat_idx}",
        representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=False)
    )
    if not feature["selected"]:
        continue
    center = feature["center"]
    size = feature["size"]
    total_volume = size * voxel_volume
    radii = (3 * total_volume / (4 * np.pi)) ** (1 / 3)
    nw.add_spheres(
        coords=center,
        colors=field_colors[feature["type"].lower()],
        radii=radii,
        name=f"{feature["type"]}_{feat_idx}_center",
        representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
    )

# nw.add_representation_within_radius_of_selection(representation_type="spacefill")
nw.display(gui=True)

In [ ]:
sp.spatial.distance_matrix(coords, centers).min(axis=-1)

In [2]:
import scicoda
aat = scicoda.atom.autodock_atom_types()

In [3]:
aat

,type,element,description,hbond_acceptor,hbond_donor,hbond_count
0,H,H,Non H-bonding hydrogen,False,False,0
1,HD,H,H-bond donor hydrogen,False,True,1
2,HS,H,Spherical H-bond donor hydrogen,False,True,<NA>
3,C,C,Aliphatic carbon,False,False,0
4,A,C,Aromatic carbon,False,False,0
5,N,N,Non H-bonding nitrogen,False,False,0
6,NA,N,H-bond acceptor nitrogen,True,False,1
7,NS,N,Spherical H-bond acceptor nitrogen,True,False,<NA>
8,OA,O,H-bond acceptor oxygen,True,False,2
9,OS,O,Spherical H-bond acceptor oxygen,True,False,<NA>


In [4]:
aat.loc[aat['type'] == 'H', 'description'].iat[0]

'Non H-bonding hydrogen'

In [6]:
import numpy as np
a = np.random.rand(5,20,21,22)
b = np.random.rand(5)

In [7]:
a.shape

(5, 20, 21, 22)

In [8]:
b.shape

(5,)

(5, 20, 21, 22)

In [17]:
a[:,1,2,2] < b

array([False,  True,  True,  True,  True])

In [19]:
np.less_equal(a, b.reshape(-1, 1, 1, 1))[:, 1,2,2]

array([False,  True,  True,  True,  True])